# ReactOT-Style Baseline on HALO8 (No Masking)

This notebook mirrors the no-masking flow-matching setup from the original ReactOT baseline,
but uses the HALO8-derived dataset at `../Data/halo8_rpsb_like_all.pkl`.

**What you will see:**
- Load the HALO8 pickled dataset
- Pick the most common atom count for fixed-shape batches
- Build a Flow-EGNN conditional model
- Train with flow matching
- Evaluate midpoint vs predicted TS RMSD on a held-out split


## Notebook Outline
1. Imports and device setup
2. Load HALO8 dataset and inspect keys
3. Choose most common atom count (no masking)
4. Train/test split + batch builder + flow-matching targets
5. Model definition (Flow-EGNN)
6. Training + held-out RMSD evaluation


In [1]:
# Core imports and device selection.
# Note: This cell prepares data or intermediate tensors used by subsequent cells.
import pickle
import math
import numpy as np
import torch
import torch.nn as nn

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Using device:", device)


Using device: mps


In [2]:
# Load the pickled HALO8-style dataset.
# Note: This visualization cell is optional and does not change model parameters.
from pathlib import Path

candidate_paths = [
    Path("../Data/halo8_rpsb_like_all.pkl"),  # when running from Notebooks/
    Path("Data/halo8_rpsb_like_all.pkl"),     # when running from repo root
]

for cpath in candidate_paths:
    if cpath.exists():
        pkl_path = cpath
        break
else:
    raise FileNotFoundError("Could not locate halo8_rpsb_like_all.pkl in expected paths.")

with open(pkl_path, "rb") as f:
    obj = pickle.load(f)

# Quick sanity check for available keys.
print("Loaded:", pkl_path)
print(obj.keys())
print(obj["reactant"].keys())

z_key = "atomic_numbers" if "atomic_numbers" in obj["reactant"] else "charges"
print("Using atom identity key:", z_key)


Loaded: ../Data/halo8_rpsb_like_all.pkl
dict_keys(['reactant', 'transition_state', 'product', 'single_fragment', 'use_ind', 'ts_guess', 'ts_guess_sbv1', 'ts_guess_true', 'ts_guess_NEBCI-xtb'])
dict_keys(['num_atoms', 'charges', 'fragments', 'positions', 'rxn', 'wB97x_6-31G(d).energy', 'wB97x_6-31G(d).atomization_energy', 'wB97x_6-31G(d).forces', 'formula', 'xtb_positions'])
Using atom identity key: charges


## Fixed Atom Count (No Masking)
ReactOT uses fixed-size tensors. We pick the most common atom count so we can batch without masking.


In [3]:
# No masking: select the most common N and create a train/test split.
# Note: This cell controls the optimization loop and key training hyperparameters.
num_samples = len(obj["reactant"]["positions"])
Ns = [obj["reactant"]["positions"][i].shape[0] for i in range(num_samples)]
from collections import Counter
cnt = Counter(Ns)
common_N, common_count = cnt.most_common(1)[0]
print("Most common N:", common_N, "count:", common_count)

idxs = [i for i in range(num_samples)
        if obj["reactant"]["positions"][i].shape[0] == common_N]
print("Usable reactions with fixed N:", len(idxs))

rng = np.random.default_rng(0)
rng.shuffle(idxs)
split = int(0.9 * len(idxs))
train_idxs = idxs[:split]
test_idxs = idxs[split:]
print(f"Train/Test sizes: {len(train_idxs)}/{len(test_idxs)}")


Most common N: 13 count: 708
Usable reactions with fixed N: 708
Train/Test sizes: 637/71


In [4]:
# Note: This helper function is defined once here and reused in later workflow steps.
def get_batch(indices, batch_size=16, rng=None):
    """Fetch a random batch of reactant/product/TS positions and atom identities."""
    if rng is None:
        rng = np.random.default_rng()
    replace = len(indices) < batch_size
    sel = rng.choice(indices, size=batch_size, replace=replace)

    xR = np.stack([obj["reactant"]["positions"][i] for i in sel], axis=0)
    xP = np.stack([obj["product"]["positions"][i] for i in sel], axis=0)
    xT = np.stack([obj["transition_state"]["positions"][i] for i in sel], axis=0)
    z  = np.stack([np.asarray(obj["reactant"][z_key][i], dtype=np.int64) for i in sel], axis=0)

    xR = torch.tensor(xR, dtype=torch.float32, device=device)
    xP = torch.tensor(xP, dtype=torch.float32, device=device)
    xT = torch.tensor(xT, dtype=torch.float32, device=device)
    z  = torch.tensor(z,  dtype=torch.long,   device=device)
    return xR, xP, xT, z

xR, xP, xT, z = get_batch(train_idxs, batch_size=16)
print(xR.shape, xP.shape, xT.shape, z.shape)


torch.Size([16, 13, 3]) torch.Size([16, 13, 3]) torch.Size([16, 13, 3]) torch.Size([16, 13])


In [5]:
# Note: This helper function is defined once here and reused in later workflow steps.
def make_flow_batch(xR, xP, xT, z):
    """Construct flow-matching targets and conditioning context."""
    x0 = 0.5 * (xR + xP)
    x1 = xT
    B  = xR.shape[0]
    t  = torch.rand(B, device=xR.device)
    x_t = (1 - t)[:, None, None] * x0 + t[:, None, None] * x1
    v_star = x1 - x0
    cond = {"z": z, "xR_ctx": xR, "xP_ctx": xP}
    return x_t, t, cond, v_star

x_t, t, cond, v_star = make_flow_batch(xR, xP, xT, z)
print("x_t", x_t.shape, "t", t.shape, "v*", v_star.shape)
print("t min/max:", float(t.min()), float(t.max()))


x_t torch.Size([16, 13, 3]) t torch.Size([16]) v* torch.Size([16, 13, 3])
t min/max: 0.04344010353088379 0.932886004447937


## Flow-EGNN Model
The model predicts a velocity field v(x,t) conditioned on reactant/product context and atomic numbers.


In [6]:
# Note: This class is a reusable building block used by later training/inference cells.
class TimeEmbedding(nn.Module):
    """Fourier-style time embedding used by the Flow-EGNN."""
    def __init__(self, n_freq=16):
        super().__init__()
        self.register_buffer("freqs", 2 * math.pi * (2 ** torch.arange(n_freq).float()))
    def forward(self, t):
        """Embed scalar times into sin/cos features."""
        args = t[:, None] * self.freqs[None, :]
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

te = TimeEmbedding(8).to(device)
out = te(t)
print(out.shape)
print(out[0, :10])


torch.Size([16, 16])
tensor([-0.6654,  0.9934,  0.2276, -0.4433, -0.7947, -0.9647,  0.5078, -0.8749,
        -0.7465,  0.1146], device='mps:0')


In [7]:
# Note: This class is a reusable building block used by later training/inference cells.
class EGNNLayer(nn.Module):
    """Single EGNN layer that updates features and coordinates."""
    def __init__(self, h_dim):
        super().__init__()
        self.phi_m = nn.Sequential(
            nn.Linear(2*h_dim + 1, h_dim),
            nn.SiLU(),
            nn.Linear(h_dim, h_dim),
            nn.SiLU(),
        )
        self.phi_h = nn.Sequential(
            nn.Linear(h_dim, h_dim),
            nn.SiLU(),
            nn.Linear(h_dim, h_dim),
        )
        self.phi_x = nn.Sequential(
            nn.Linear(h_dim, h_dim),
            nn.SiLU(),
            nn.Linear(h_dim, 1),
        )

    def forward(self, h, x, src, dst):
        """Compute message passing and coordinate updates."""
        B, N, H = h.shape
        xi = x[:, src, :]
        xj = x[:, dst, :]
        rij = xi - xj
        dij = (rij ** 2).sum(dim=-1, keepdim=True)

        hi = h[:, src, :]
        hj = h[:, dst, :]
        mij = self.phi_m(torch.cat([hi, hj, dij], dim=-1))

        agg = torch.zeros((B, N, H), device=h.device, dtype=h.dtype)
        agg.scatter_add_(1, src[None, :, None].expand(B, -1, H), mij)

        h = h + self.phi_h(agg)

        s = self.phi_x(mij)
        dx_ij = rij * s
        dx = torch.zeros((B, N, 3), device=x.device, dtype=x.dtype)
        dx.scatter_add_(1, src[None, :, None].expand(B, -1, 3), dx_ij)
        x = x + dx
        return h, x

def fully_connected_edges(N, device):
    """Return source/target indices for a fully-connected graph."""
    idx = torch.arange(N, device=device)
    ii, jj = torch.meshgrid(idx, idx, indexing="ij")
    mask = ii != jj
    src = ii[mask].reshape(-1)
    dst = jj[mask].reshape(-1)
    return src, dst


In [8]:
# Note: This class is a reusable building block used by later training/inference cells.
class FlowEGNN(nn.Module):
    """Flow-EGNN that predicts velocity fields conditioned on context."""
    def __init__(self, z_max=100, h_dim=128, n_layers=4, time_freq=16, use_concat_ctx=True):
        super().__init__()
        self.use_concat_ctx = use_concat_ctx
        self.z_emb = nn.Embedding(z_max+1, h_dim)

        self.t_emb = TimeEmbedding(n_freq=time_freq)
        self.t_proj = nn.Linear(2*time_freq, h_dim)

        if use_concat_ctx:
            self.ctx_proj = nn.Linear(9, h_dim)
        else:
            self.ctx_proj = nn.Linear(3, h_dim)

        self.layers = nn.ModuleList([EGNNLayer(h_dim) for _ in range(n_layers)])

        self.v_head = nn.Sequential(
            nn.Linear(h_dim, h_dim),
            nn.SiLU(),
            nn.Linear(h_dim, 3)
        )

    def forward(self, x_t, t, cond):
        """Predict per-atom velocity v(x,t) for the current positions."""
        z = cond["z"]
        xR = cond["xR_ctx"]
        xP = cond["xP_ctx"]

        B, N, _ = x_t.shape
        h = self.z_emb(z)

        te = self.t_proj(self.t_emb(t))
        h = h + te[:, None, :].expand(B, N, -1)

        diff = xP - xR
        if self.use_concat_ctx:
            ctx = torch.cat([xR, xP, diff], dim=-1)
        else:
            ctx = diff
        h = h + self.ctx_proj(ctx)

        src, dst = fully_connected_edges(N, x_t.device)
        x = x_t
        for layer in self.layers:
            h, x = layer(h, x, src, dst)

        v = self.v_head(h)
        return v


In [9]:
# Quick sanity check of model outputs.
# Note: This cell prepares data or intermediate tensors used by subsequent cells.
model_diff = FlowEGNN(h_dim=128, n_layers=3, use_concat_ctx=False).to(device)
model_cat  = FlowEGNN(h_dim=128, n_layers=3, use_concat_ctx=True ).to(device)

v1 = model_diff(x_t, t, cond)
v2 = model_cat(x_t, t, cond)

print(v1.shape, v2.shape)
print("v1 norm mean:", float(v1.norm(dim=-1).mean()))
print("v2 norm mean:", float(v2.norm(dim=-1).mean()))


torch.Size([16, 13, 3]) torch.Size([16, 13, 3])
v1 norm mean: 0.5927435755729675
v2 norm mean: 1.0267804861068726


/var/folders/8p/6t_fjn5d1c7fxwflz29kdvqr0000gn/T/ipykernel_10496/3189143095.py:9: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:837.)
  print("v1 norm mean:", float(v1.norm(dim=-1).mean()))


In [10]:
# Note: This helper function is defined once here and reused in later workflow steps.
def flow_loss(v_pred, v_star):
    """Mean-squared error loss for flow matching."""
    return ((v_pred - v_star) ** 2).mean()

opt = torch.optim.Adam(model_cat.parameters(), lr=1e-3)

opt.zero_grad()
v_pred = model_cat(x_t, t, cond)
loss = flow_loss(v_pred, v_star)
loss.backward()

grad_norms = []
for name, p in model_cat.named_parameters():
    if p.grad is not None:
        grad_norms.append((name, float(p.grad.norm().detach().cpu())))
print("loss:", float(loss.detach().cpu()))
print("some grad norms:", grad_norms[:5])


loss: 0.5024742484092712
some grad norms: [('z_emb.weight', 0.358839750289917), ('t_proj.weight', 0.8382219076156616), ('t_proj.bias', 0.5717273950576782), ('ctx_proj.weight', 2.8881466388702393), ('ctx_proj.bias', 0.5717273354530334)]


## Training
A short training loop to show loss decreasing. This is intentionally small for clarity.


In [11]:
# Note: This cell controls the optimization loop and key training hyperparameters.
model = model_cat
opt = torch.optim.Adam(model.parameters(), lr=3e-4)

losses = []
for step in range(200):
    # Sample a new training batch each step.
    xR, xP, xT, z = get_batch(train_idxs, batch_size=32)
    x_t, t, cond, v_star = make_flow_batch(xR, xP, xT, z)

    opt.zero_grad()
    v_pred = model(x_t, t, cond)
    loss = ((v_pred - v_star)**2).mean()
    loss.backward()
    opt.step()

    losses.append(float(loss.detach().cpu()))
    if step % 20 == 0:
        print(step, losses[-1])


0 0.5307310223579407
20 0.06970393657684326
40 0.05198609456419945
60 0.05373501777648926
80 0.06782344728708267
100 0.04685414955019951
120 0.05232061445713043
140 0.05449623987078667
160 0.05162009224295616
180 0.0488412007689476


## ODE Integration and Held-Out RMSD
We integrate the learned velocity field from midpoint to TS and compare midpoint vs prediction RMSD on held-out reactions.


In [12]:
# Note: Metrics in this cell are for validation/diagnostics rather than parameter updates.
@torch.no_grad()
def euler_solve(model, x0, cond, steps=50):
    """Integrate dx/dt = v(x,t) with a simple Euler solver."""
    x = x0
    dt = 1.0 / steps
    for k in range(steps):
        t = torch.full((x.shape[0],), k * dt, device=x.device)
        v = model(x, t, cond)
        x = x + dt * v
    return x

def rmsd_torch(A, B):
    """Compute per-structure Kabsch-aligned RMSD over atoms."""
    A_centered = A - A.mean(dim=1, keepdim=True)
    B_centered = B - B.mean(dim=1, keepdim=True)
    cov = A_centered.transpose(1, 2) @ B_centered
    U, _, Vh = torch.linalg.svd(cov)
    det = torch.sign(torch.linalg.det(U @ Vh))
    corr = torch.eye(3, device=A.device, dtype=A.dtype).unsqueeze(0).repeat(A.shape[0], 1, 1)
    corr[:, -1, -1] = det
    rot = U @ corr @ Vh
    A_aligned = A_centered @ rot
    return torch.sqrt(((A_aligned - B_centered)**2).sum(dim=-1).mean(dim=-1))

@torch.no_grad()
def evaluate_rmsd(model, indices, batch_size=32, steps=80):
    midpoint_rmsd = []
    pred_rmsd = []

    for start in range(0, len(indices), batch_size):
        chunk = indices[start:start + batch_size]
        if len(chunk) == 0:
            continue

        # Deterministic chunk sampling for evaluation.
        xR = np.stack([obj["reactant"]["positions"][i] for i in chunk], axis=0)
        xP = np.stack([obj["product"]["positions"][i] for i in chunk], axis=0)
        xT = np.stack([obj["transition_state"]["positions"][i] for i in chunk], axis=0)
        z  = np.stack([np.asarray(obj["reactant"][z_key][i], dtype=np.int64) for i in chunk], axis=0)

        xR = torch.tensor(xR, dtype=torch.float32, device=device)
        xP = torch.tensor(xP, dtype=torch.float32, device=device)
        xT = torch.tensor(xT, dtype=torch.float32, device=device)
        z  = torch.tensor(z, dtype=torch.long, device=device)

        x0 = 0.5 * (xR + xP)
        cond = {"z": z, "xR_ctx": xR, "xP_ctx": xP}
        xTS_pred = euler_solve(model, x0, cond, steps=steps)

        midpoint_rmsd.append(rmsd_torch(x0, xT).detach().cpu())
        pred_rmsd.append(rmsd_torch(xTS_pred, xT).detach().cpu())

    midpoint_rmsd = torch.cat(midpoint_rmsd)
    pred_rmsd = torch.cat(pred_rmsd)
    return midpoint_rmsd, pred_rmsd

mid_rmsd, pred_rmsd = evaluate_rmsd(model, test_idxs, batch_size=32, steps=80)
improve = mid_rmsd - pred_rmsd

print(f"Held-out reactions: {len(test_idxs)}")
print(f"Midpoint RMSD mean/median: {mid_rmsd.mean():.4f} / {mid_rmsd.median():.4f}")
print(f"Pred RMSD mean/median    : {pred_rmsd.mean():.4f} / {pred_rmsd.median():.4f}")
print(f"Pred RMSD p90            : {torch.quantile(pred_rmsd, 0.9):.4f}")
print(f"Mean RMSD improvement    : {improve.mean():.4f}")
print(f"% improved vs midpoint   : {(improve > 0).float().mean() * 100:.1f}%")


Held-out reactions: 71
Midpoint RMSD mean/median: 0.3329 / 0.2830
Pred RMSD mean/median    : 0.3298 / 0.3122
Pred RMSD p90            : 0.5618
Mean RMSD improvement    : 0.0031
% improved vs midpoint   : 49.3%
